In [ ]:
from time import sleep as delay
from project.el5600 import project
if "ic" in globals():
    ic.close()
ic = project(device="el5600", revision="a1", emulator="cp2112", logging=False, is_gui=False)

from phy.multimeter import siglent_sdm3055_auto
from phy.power_supply import keithley_2460, rigol_dp821a, keysight_e36232a
from phy.scope import tektronix_dpo4104b_usb
from phy.eloader import sdl1020x
from phy.relay_16ch import relay_box
# from phy.chamber_th3 import chamber

%matplotlib tk
from interface.docs.output_chart import plot
from interface.cui_colors import color
from interface.i2c_bridge.tca9548a import tca9548
from interface.docs.output_excel import excel_frame, style
from interface.cui_logger import logger as log

import numpy as np

chart = plot()

dm = siglent_sdm3055_auto()
ps = rigol_dp821a()
ks = keysight_e36232a()
kt = keithley_2460()
ld = sdl1020x()
sc = tektronix_dpo4104b_usb()

# dma = agilent_34401a("COM5")
relay = relay_box(i2c_h=ic)
# tc = chamber(port=3)
# mux = tca9548(ic.i2c_h, 0x70)


# ==================================
def disable_all_ps(kt=kt, ps=ps):
    kt.disable
    ks.disable
    ld.disable
    ps.ch1.disable
    ps.ch2.disable
# ==================================

disable_all_ps()

# Test A6

kt : ---> VDD3V3

ks : direct to VIN

ps.ch1 : ---> relay power

ps.ch2 : ---> relay.ch3 (EN)

relay.ch1 : VIN

relay.ch2 : VOUT scope monitoring

relay.ch3 : EN

loader : disconnect

In [ ]:
temperature = "N40C"

log.output_set_filename(f"[{temperature}_6] VIN_OVP, V_OVP_HYS")
log.output_csv(["VIN (V)", "VDD3V3 (V)", "EN (V)", "VIN_POK_STS", "SW_STS", "VIN_OVP_TH", "VIN_OVP_EN", "VIN_OV_AUTO_ON_EN"])

In [ ]:
disable_all_ps()
relay.init_channels

dict_vin_ovp = {
    0 : 6,
    1 : 11,
    2 : 15,
    3 : 18,
    4 : 22.5,
    5 : 23.5,
    6 : 31.5,
    7 : 33
}

v_vdd = 3.3
v_en = 3.3

# --------------------------------------------
ps.ch1.cfg_all = 5, 1 # relay
ps.ch1.enable

# --------------------------------------------
kt.cfg_all = v_vdd, 0.01 # vdd
kt.enable
delay(0.5)

ps.ch2.cfg_all = 3.3, 0.1
ps.ch2.enable
delay(0.5)
relay.ch3.enable # en
delay(0.5)

ks.cfg_all = dict_vin_ovp[0]*0.8, 0.5 # vin
ks.enable
delay(0.5)

relay.ch1.enable # vin
relay.ch2.enable # for vout monitoring
delay(0.5)

ic.VIN_OVP_TH = 0
ic.VIN_OVP_EN = 1
ic.VIN_OV_AUTO_ON_EN = 1

In [ ]:
# ----------------------------------------------------------------------------

vin_ovp_en = ic.VIN_OVP_EN
vin_ov_auto_on_en = ic.VIN_OV_AUTO_ON_EN

for ov_index, ov_th in dict_vin_ovp.items():
    
    ic.VIN_OVP_TH = ov_index
    
    v_start  = ov_th * 0.8
    v_finish = ov_th * 1.2

    list_temp = list(np.arange(v_start, v_finish, 0.01))
    list_vin_ov = [round(num, 2) for num in list_temp]
    
    for vin in list_vin_ov:
        
        if vin > 40:
            vin = 39.9
            
        ks.vset = vin
        
        try:
            vin_pok_sts = ic.VIN_POK_STS
            sw_sts = ic.SW_STS
        except:
            vin_pok_sts = "NACK"
            sw_sts = "NACK"
        
        ret = [vin, v_vdd, v_en, vin_pok_sts, sw_sts, ov_index, vin_ovp_en, vin_ov_auto_on_en]
        log.output_csv(ret)
        
        print(ret)
    
    for vin in reversed(list_vin_ov):
        
        ks.vset = vin
        
        try:
            vin_pok_sts = ic.VIN_POK_STS
            sw_sts = ic.SW_STS
        except:
            vin_pok_sts = "NACK"
            sw_sts = "NACK"
        
        ret = [vin, v_vdd, v_en, vin_pok_sts, sw_sts, ov_index, vin_ovp_en, vin_ov_auto_on_en]
        log.output_csv(ret)
        
        print(ret)

# ----------------------------------------------------------------------------

relay.init_channels
disable_all_ps()